In [ ]:
import sys
sys.path.insert(0, "/Users/bohaoli/Desktop/tuto/tuto_langchain/official/langchain-langgraph-langsmith/utils")

from utils import ppm, ppms

## LangChain vs LangGraph: Message Return Types

### 1. Single `AIMessage` object
**When:** Calling a LangChain model directly.
```python
result = model.invoke([HumanMessage(content="Hi")])
# result is AIMessage(content='Hello!', ...)
```

### 2. A list of messages (`[HumanMessage, AIMessage, ...]`)
**When:** You build the message list yourself, or extract it from graph state manually.
```python
messages = [HumanMessage(content="Hi"), AIMessage(content="Hello!")]
# or
messages = graph.invoke(input)["messages"]
```

### 3. A dict with `"messages"` key (`{"messages": [...]}`)
**When:** Calling `graph.invoke()` in LangGraph. It returns the full graph **state**, which is a dict. The `messages` key matches whatever you defined in your `State` class.
```python
result = graph.invoke({"messages": [HumanMessage(content="Hi")]})
# result is {"messages": [HumanMessage(...), AIMessage(...)]}
```

The `ppm` and `ppms` functions below handle all three cases.

In [ ]:
import sys
sys.path.insert(0, "/Users/bohaoli/Desktop/tuto/tuto_langchain/official/langchain-langgraph-langsmith/utils")

from utils import ppm, ppms

In [ ]:
import sys
sys.path.append('/Users/bohaoli/Desktop/tuto/tuto_langchain/official/001-foundation-langraph')
import importlib
import printer
importlib.reload(printer)
from printer import display_langchain_messages_dark  # noqa: E402
# display_langchain_messages_dark(response["messages"])

## Langfuse Tracing (instead of LangSmith)

Pass `langfuse_handler` as a callback to trace LangChain/LangGraph calls to your local Langfuse instance.

```python
# For a model call
result = llm.invoke(messages, config={"callbacks": [langfuse_handler]})

# For a graph call
result = graph.invoke({"messages": messages}, config={"callbacks": [langfuse_handler]})
```

In [4]:
from dotenv import load_dotenv
load_dotenv()

from langfuse.langchain import CallbackHandler
langfuse_handler = CallbackHandler()

# Verify connection to Langfuse
from langfuse import Langfuse
Langfuse().auth_check()  # should print True if connected

True

### Sample: LangChain model call traced to Langfuse

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o")

# This call will appear as a trace in Langfuse
result = llm.invoke(
    [HumanMessage(content="What is 2 + 2?")],
    config={"callbacks": [langfuse_handler]}
)
print(result.content)

2 + 2 equals 4.


### Sample: LangGraph agent call traced to Langfuse

In [ ]:
from langgraph.prebuilt import create_react_agent

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

agent = create_react_agent(llm, [multiply])

# This call will appear as a trace in Langfuse
result = agent.invoke(
    {"messages": [HumanMessage(content="What is 6 times 7?")]},
    config={"callbacks": [langfuse_handler]}
)

for m in result["messages"]:
    m.pretty_print()

Now open http://localhost:3000 and check the **Traces** tab — you should see both calls above.